# 分子系統解析入門2　〜鯨が歩んだ道をたどろう〜

## 概要
**38種類の哺乳類**(海生・半海生の哺乳類37種 + 外群としてヒト)のミトコンドリアDNAから系統樹を作り、「海に戻った哺乳類」の進化を読み解きます。

## このデータセットについて
鳥類のデータセットと違い、**海生哺乳類とその陸上の親戚** にフォーカスしています。
系統樹の根を決めるための **外群 (outgroup)** として、これらから系統的に離れた **ヒト**(霊長目)を1種加えてあります。

| 系統 | 種数 | 例 |
| --- | --- | --- |
| **Mysticeti**(ヒゲクジラ亜目) | 7 | シロナガスクジラ、ザトウクジラ、コククジラなど |
| **Odontoceti**(ハクジラ亜目) | 15 | マッコウ、シャチ、イルカ、イッカク、カワイルカなど |
| **Pinnipedia**(鰭脚類) | 6 | アザラシ、アシカ、トド、セイウチ |
| **Sirenia**(海牛目) | 2 | マナティー、ジュゴン |
| **Mustelidae**(イタチ科) | 1 | ラッコ |
| **Ursidae**(クマ科) | 1 | ホッキョクグマ |
| **Terrestrial-sister**(陸上の姉妹群) | 5 | カバ、ヒグマ、カワウソ、ゾウ、ハイラックス |
| **Outgroup**(外群) | 1 | ヒト(霊長目) |

## 解析の流れ

1. **入門編** — Pythonだけで完結する単純なアプローチ(切り詰めアラインメント + 近隣結合法)
2. **本格編** — **MAFFT** + **IQ-TREE** で最尤法による系統樹推定を実行

## ライセンス
- 配列データ：NCBIのパブリックデータ
- シルエット画像：PhyloPic(CC ライセンス、`mammals/phylopic_mammals/phylopic_attribution.tsv` 参照)

## 0. 必要ライブラリのインストール(初回のみ)

Python パッケージ:
- `biopython` : 配列解析と系統樹計算
- `matplotlib-fontja` : 日本語フォント
- `ete3` + `PyQt5` : 系統樹の描画(このノートブックの樹はすべて ETE3 で描きます)

**MAFFT と IQ-TREE は別途インストールが必要です(後のセクション 12 で扱います)。**

> Note: ETE3 は内部で Qt を使って樹を画像に描き出します。画面のないサーバーや Colab で動かす場合は、
> 次のセルで環境変数 `QT_QPA_PLATFORM=offscreen` を設定して **ヘッドレス描画** に切り替えます。

In [ ]:
%pip install biopython pandas numpy matplotlib seaborn matplotlib-fontja ete3 PyQt5

## 1. ライブラリの読み込みと日本語フォントの設定

**順序が重要**:seaborn のスタイル設定を先に行い、その後に `matplotlib_fontja` を読み込みます。

In [ ]:
import os
import sys
import shutil
import tempfile
import subprocess
import warnings
from pathlib import Path

# ETE3 はヘッドレス環境(画面なしのサーバー・Colab)でも描画できるよう、
# Qt のインポートより前に offscreen プラットフォームを指定しておく
os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.colors import to_rgb
from matplotlib.patches import Patch
import seaborn as sns

from Bio import SeqIO, AlignIO, Phylo
from Bio.Align import MultipleSeqAlignment
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor

# 系統樹の描画は ETE3 で行う
from ete3 import Tree as EteTree, TreeStyle, TextFace, ImgFace, NodeStyle, RectFace
from IPython.display import Image as IPyImage, display

# 日本語フォントの設定(順序が重要)
sns.set_style("whitegrid")
try:
    import matplotlib_fontja  # noqa: F401
    _sans = [f for f in plt.rcParams["font.sans-serif"] if f != "IPAexGothic"]
    plt.rcParams["font.sans-serif"] = ["IPAexGothic"] + _sans
    print("matplotlib-fontja: OK")
except ImportError:
    print("matplotlib-fontja が見つかりません。上のセルでインストールしてください。")
plt.rcParams["axes.unicode_minus"] = False

# ETE3 の TextFace で日本語を表示するためのフォント名(CJK を含むものを指定)
JP_FONT = "Noto Sans CJK JP"

# データファイルへのパス(このノートブックは ml/ から実行する想定)
MAMMALS_DIR = Path("mammals")
FASTA_PATH = MAMMALS_DIR / "mammals_cytb.fasta"
METADATA_PATH = MAMMALS_DIR / "mammals_cytb_metadata.tsv"
PHYLOPIC_DIR = MAMMALS_DIR / "phylopic_mammals" / "png"

# MAFFT / IQ-TREE 用の作業ディレクトリ
WORK_DIR = Path("./phylo_work_mammals")
WORK_DIR.mkdir(exist_ok=True)

for p in [FASTA_PATH, METADATA_PATH, PHYLOPIC_DIR]:
    assert p.exists(), f"見つかりません: {p}"
print(f"データ準備 OK: {MAMMALS_DIR.resolve()}")
print(f"作業ディレクトリ: {WORK_DIR.resolve()}")

## 2. FASTA ファイルの読み込み

**FASTA** は配列データを表す基本的なテキスト形式です。

```
>Balaena_mysticetus|NC_005268.1|CYTB    ← ヘッダー
ATGACCAACATCCGAAAAACACAC...              ← 配列本体
```

ヘッダーは `>属_種|NCBI アクセッション番号|遺伝子名` という形式です。

In [ ]:
# データファイルへのパス(このノートブックは ml/ から実行する想定)
MAMMALS_DIR = Path("mammals")
FASTA_PATH = MAMMALS_DIR / "mammals_cytb.fasta"
METADATA_PATH = MAMMALS_DIR / "mammals_cytb_metadata.tsv"
PHYLOPIC_DIR = MAMMALS_DIR / "phylopic_mammals" / "png"

# MAFFT / IQ-TREE 用の作業ディレクトリ
WORK_DIR = Path("./phylo_work_mammals")
WORK_DIR.mkdir(exist_ok=True)

for p in [FASTA_PATH, METADATA_PATH, PHYLOPIC_DIR]:
    assert p.exists(), f"見つかりません: {p}"
print(f"データ準備 OK: {MAMMALS_DIR.resolve()}")
print(f"作業ディレクトリ: {WORK_DIR.resolve()}")

records = list(SeqIO.parse(FASTA_PATH, "fasta"))
print(f"読み込んだ配列数: {len(records)}")
print(f"最初のヘッダー : {records[0].id}")
print(f"最初の60塩基   : {str(records[0].seq)[:60]} ...")

# ヘッダーから「種名(属_種)」だけを取り出して ID を整理
for rec in records:
    rec.id = rec.id.split("|")[0]
    rec.description = ""

print(f"\n整理後の ID 一覧(先頭5件): {[r.id for r in records[:5]]}")

## 3. メタデータ（和名・分類群）の読み込み

各種の和名・英名・分類群が `mammals_cytb_metadata.tsv` にまとめられています。
> - 同じ「**Cetacea(クジラ目)**」でも、**Mysticeti**(ヒゲクジラ)と **Odontoceti**(ハクジラ)に分かれる
> - 「**Carnivora(食肉目)**」の中に、海(Pinnipedia, ラッコ)・氷上(ホッキョクグマ)・陸(ヒグマ、カワウソ)が混在
> - 「**Terrestrial-sister**」は、海生グループの**陸上の最近縁種**として明示的に選ばれている(例:カバはクジラの姉妹群)

In [ ]:
metadata = pd.read_csv(METADATA_PATH, sep="\t")
metadata["id"] = metadata["scientific_name"].str.replace(" ", "_")

print(f"メタデータ行数: {len(metadata)}")
print("\n各系統の種数:")
print(metadata["clade"].value_counts())
print("\n各目 (order) の種数:")
print(metadata["order"].value_counts())
metadata[["scientific_name", "common_name_ja", "order", "clade"]].head(10)

## 4. 配列の長さと組成を確認

- **配列長** がそろっているか? → そろっていなければ後でアラインメントが必要
- **GC 含量** (G + C の割合) → 系統で傾向があるか?

In [ ]:
def gc_content(seq):
    s = str(seq).upper()
    return (s.count("G") + s.count("C")) / len(s) * 100

seq_stats = pd.DataFrame({
    "id": [r.id for r in records],
    "length": [len(r.seq) for r in records],
    "gc_pct": [gc_content(r.seq) for r in records],
}).merge(metadata[["id", "common_name_ja", "clade"]], on="id", how="left")

print("配列長の統計:")
print(seq_stats["length"].describe().round(1))
print(f"\n配列長の種類: {sorted(seq_stats['length'].unique())}")
seq_stats.head()

In [ ]:
# 系統別の GC 含量(箱ひげ図 + 個々の点)
fig, ax = plt.subplots(figsize=(11, 4))
order = seq_stats.groupby("clade")["gc_pct"].median().sort_values().index
sns.boxplot(data=seq_stats, x="clade", y="gc_pct", order=order,
            color="lightsteelblue", ax=ax)
sns.stripplot(data=seq_stats, x="clade", y="gc_pct", order=order,
              color="darkblue", alpha=0.6, size=4, ax=ax)
ax.set_xlabel("系統 (clade)")
ax.set_ylabel("GC 含量 (%)")
ax.set_title("cytb 遺伝子の GC 含量(系統別)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 5. 入門編 — シンプルなアラインメント

DNA から系統樹を作るには、**「同じ位置に同じ進化的起源の塩基を縦に揃える」** 作業 = **マルチプルアラインメント (MSA)** が必要です。

今回のデータには都合のいい特徴があります。

- すべての配列が **ATG**(開始コドン)から始まっている
- 長さは 1137 〜 1141 塩基とほぼ同じ(脊椎動物の cytb は長さがよく保存されている)

入門編では、簡単のため **全配列の先頭から共通の長さだけを切り出して使う** ことにします。
本格的なアラインメントは後のセクション 12 で MAFFT を使って行います。

### 5.1 切り詰めアライメント

In [ ]:
min_len = min(len(r.seq) for r in records)
print(f"全配列を {min_len} 塩基に切り詰めます")

for rec in records:
    rec.seq = rec.seq[:min_len]

alignment = MultipleSeqAlignment(records)
print(f"\nアラインメント:")
print(f"  配列数 : {len(alignment)}")
print(f"  カラム数: {alignment.get_alignment_length()}")

### 5.2 進化距離の計算

**p距離** で種同士の進化的な「離れぐあい」を測ります。

$$d_{\mathrm{p}}(A, B) = \frac{\text{異なる塩基の数}}{\text{全塩基数}}$$

In [ ]:
calculator = DistanceCalculator("identity")
dm = calculator.get_distance(alignment)

print(f"距離行列のサイズ: {len(dm.names)} × {len(dm.names)}")
print(f"\n注目の比較:")
print(f"  シロナガス vs ザトウ(どちらもヒゲクジラ)")
print(f"    d = {dm['Balaenoptera_musculus', 'Megaptera_novaeangliae']:.4f}")
print(f"  シロナガス vs マッコウ(ヒゲ vs ハクジラ)")
print(f"    d = {dm['Balaenoptera_musculus', 'Physeter_macrocephalus']:.4f}")
print(f"  シロナガス vs カバ(クジラとカバは姉妹群)")
print(f"    d = {dm['Balaenoptera_musculus', 'Hippopotamus_amphibius']:.4f}")
print(f"  ホッキョクグマ vs ヒグマ(近縁種同士)")
print(f"    d = {dm['Ursus_maritimus', 'Ursus_arctos']:.4f}")
print(f"  マナティー vs アフリカゾウ(マナティーとゾウは同じアフリカ獣類)")
print(f"    d = {dm['Trichechus_manatus', 'Loxodonta_africana']:.4f}")

### 5.3 距離行列をヒートマップで可視化

行と列を **系統(clade)順** に並べると、近縁な種同士が「ブロック」を成して見えるはずです。

In [ ]:
names = list(dm.names)
n = len(names)
mat = np.array([[dm[a, b] for b in names] for a in names])
dist_df = pd.DataFrame(mat, index=names, columns=names)

# clade 順に並べ替え
clade_order_list = [
    "Mysticeti", "Odontoceti",          # クジラ
    "Pinnipedia", "Mustelidae", "Ursidae",  # 食肉目の海生・半海生
    "Sirenia",                          # 海牛目
    "Terrestrial-sister",               # 陸上姉妹群
    "Outgroup",                         # 外群(ヒト)
]
id_to_clade = metadata.set_index("id")["clade"].to_dict()
ordered = sorted(names, key=lambda x: (clade_order_list.index(id_to_clade.get(x, "Terrestrial-sister")), x))
dist_df = dist_df.loc[ordered, ordered]

ja_map = metadata.set_index("id")["common_name_ja"].to_dict()
labels_ja = [ja_map.get(x, x) for x in dist_df.index]

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(dist_df, cmap="viridis",
            xticklabels=labels_ja, yticklabels=labels_ja,
            cbar_kws={"label": "同一性距離"}, ax=ax, square=True)
ax.set_title("哺乳類38種間の遺伝的距離(系統順に並べ替え)")
plt.tight_layout()
plt.show()

**観察ポイント**

- クジラ目(Mysticeti + Odontoceti)はひとつの大きな「濃いブロック」を作っている
- 鰭脚類(Pinnipedia)も互いに近い
- でも **クジラ全体 vs 鰭脚類** はかなり離れている(別系統の海生哺乳類)
- ラッコ(Mustelidae)は **カワウソ**(Terrestrial-sister)と近い色をしているはず

### 5.4 近隣結合法（NJ法）で系統樹を作る

系統樹の「根（最も古い分岐点）」を決めるには、解析対象から系統的に離れていると分かっている種 = **外群 (outgroup)** が必要です。今回はその外群として **ヒト** を加えてあります。ヒトでルートし直すことで、系統樹が「海・陸の哺乳類 vs ヒト」という構造になり、どの分岐が根に近いかがはっきりします。

In [ ]:
constructor = DistanceTreeConstructor()
nj_tree = constructor.nj(dm)

# 外群(ヒト)で根を打ち直す → 系統樹が「海・陸の哺乳類 vs ヒト」の構造になる
nj_tree.root_with_outgroup("Homo_sapiens")

# 内部ノードのデフォルト名(Inner1, Inner2, ...)を消す
for clade in nj_tree.get_nonterminals():
    clade.name = None

print(f"NJ 系統樹を作成しました")
print(f"  末端ノード(種)の数: {nj_tree.count_terminals()}")

### 5.5 系統樹の描画（まずはシンプルに）

系統樹の描画には **ETE3** (`ete3`) を使います。ETE3 は系統樹の操作・可視化に特化したライブラリで、
和名ラベルや系統色、各種のシルエット画像、ブートストラップ値などを柔軟にレイアウトできます。

このセルでは、以降のすべての樹で使い回す **共通の描画ツールキット** を定義します。

- `bio_to_ete()` : Biopython の系統樹を ETE3 の `Tree` に変換(枝長・ブートストラップ値ごと引き継ぐ)
- `render_ete_to_png()` / `draw_tree_ete()` : 系統色・和名・シルエット・ブートストラップ付きで描画

まずはシルエットなしで、和名ラベルを系統別の色で塗り分けたシンプルな樹を描いてみます。

In [ ]:
# === 系統別の色の設定(海生 = 寒色系、陸生 = 暖色系)===
clade_to_color = {
    "Mysticeti":          "#1f4e79",  # 濃紺 — ヒゲクジラ
    "Odontoceti":         "#2e86ab",  # 青  — ハクジラ
    "Pinnipedia":         "#1abc9c",  # 緑青 — 鰭脚類
    "Sirenia":            "#16a085",  # 深緑 — 海牛
    "Mustelidae":         "#d35400",  # 茶  — ラッコ
    "Ursidae":            "#7d6608",  # 茶金 — ホッキョクグマ
    "Terrestrial-sister": "#7f8c8d",  # 灰 — 陸上姉妹群
    "Outgroup":           "#444444",  # 濃灰 — 外群(ヒト)
}
id_to_color = {row["id"]: clade_to_color.get(row["clade"], "#000000")
               for _, row in metadata.iterrows()}

# === ETE3 による系統樹描画ツールキット(以降のすべての樹で使い回す) ===

def bio_to_ete(bio_tree):
    """Biopython の系統樹を ETE3 の Tree に変換する。

    枝長(branch_length → dist)とブートストラップ値(confidence → support)を引き継ぐ。
    ルート(一番外側のノード)の枝は描画しないよう dist=0 にしておく。
    """
    def add(bio_clade, ete_node):
        for child in bio_clade.clades:
            n = ete_node.add_child(name=child.name or "",
                                   dist=child.branch_length or 0.0)
            if child.confidence is not None:
                n.support = child.confidence
            add(child, n)
    root = EteTree()
    root.name = bio_tree.root.name or ""
    root.dist = 0.0          # ルートの枝は除外(左端に伸びる余分な線を描かない)
    add(bio_tree.root, root)
    return root


_silhouette_cache = {}

def colored_silhouette(species, color):
    """黒シルエット PNG を系統色に塗り替えて一時ファイルに保存し、(パス, 表示幅) を返す。

    ETE3 の ImgFace はファイルパスを要求するため、色付き画像を一旦書き出して渡す。
    アスペクト比を保つよう、高さ36pxに対する幅を計算する。
    """
    key = (species, color)
    if key in _silhouette_cache:
        return _silhouette_cache[key]
    src = PHYLOPIC_DIR / f"{species}.png"
    if not src.exists():
        _silhouette_cache[key] = None
        return None
    img = mpimg.imread(src).astype(float)
    if img.max() > 1:
        img /= 255.0
    if img.ndim != 3 or img.shape[2] != 4:   # アルファ付き RGBA のみ対象
        _silhouette_cache[key] = None
        return None
    rgb = to_rgb(color)
    out = np.zeros_like(img)
    out[..., 0], out[..., 1], out[..., 2] = rgb[0], rgb[1], rgb[2]
    out[..., 3] = img[..., 3]              # アルファ(シルエットの形)は保持
    tmp_dir = Path(tempfile.gettempdir()) / "ete_silhouettes_mammals"
    tmp_dir.mkdir(exist_ok=True)
    dst = tmp_dir / f"{species}_{color.lstrip('#')}.png"
    mpimg.imsave(dst, out)
    h, w = img.shape[:2]
    result = (str(dst), max(1, int(36 * w / h)))
    _silhouette_cache[key] = result
    return result


def render_ete_to_png(tree, title, show_bootstrap=False, with_silhouettes=True,
                      width=1000, out_path=None):
    """Biopython 系統樹を ETE3 で描画して PNG に書き出し、ファイルパスを返す。

    - 葉:枝と和名ラベルを系統色で塗り、必要なら PhyloPic シルエットを並べる
    - 内部ノード:show_bootstrap=True ならブートストラップ値を支持率に応じた色で表示
    - 樹のタイトルと系統(clade)の凡例も ETE3 上で付与する
    """
    et = bio_to_ete(tree)

    def layout(node):
        if node.is_leaf():
            color = id_to_color.get(node.name, "#000000")
            ns = NodeStyle()
            ns["hz_line_color"] = color
            ns["hz_line_width"] = 2
            ns["size"] = 0
            node.set_style(ns)
            if with_silhouettes:
                sil = colored_silhouette(node.name, color)
                if sil:
                    node.add_face(ImgFace(sil[0], width=sil[1], height=36),
                                  column=0, position="aligned")
            label = ja_map.get(node.name, node.name)
            node.add_face(TextFace("  " + label, fsize=11, ftype=JP_FONT, fgcolor=color),
                          column=1, position="aligned")
        else:
            ns = NodeStyle()
            ns["hz_line_color"] = "#888888"
            ns["vt_line_color"] = "#888888"
            ns["size"] = 0
            node.set_style(ns)
            if show_bootstrap and node.support:
                bs = node.support
                bc = "#1b9e77" if bs >= 95 else ("#d95f02" if bs >= 70 else "#b22222")
                node.add_face(TextFace(f"{bs:.0f} ", fsize=8, fgcolor=bc),
                              column=0, position="branch-top")

    ts = TreeStyle()
    ts.show_leaf_name = False
    ts.layout_fn = layout
    ts.show_scale = True            # 枝長スケールバーを表示
    ts.title.add_face(TextFace(title, fsize=14, ftype=JP_FONT), column=0)

    # 系統(clade)の凡例
    for clade_name, c in clade_to_color.items():
        ts.legend.add_face(RectFace(16, 16, c, c), column=0)
        ts.legend.add_face(TextFace("  " + clade_name + "  ", fsize=10), column=1)
    if show_bootstrap:
        for lbl, c in [("BS ≥ 95", "#1b9e77"), ("BS 70–94", "#d95f02"), ("BS < 70", "#b22222")]:
            ts.legend.add_face(RectFace(16, 16, c, c), column=0)
            ts.legend.add_face(TextFace("  " + lbl + "  ", fsize=10), column=1)
    ts.legend_position = 4          # 右下

    if out_path is None:
        out_path = Path(tempfile.gettempdir()) / "ete_tree_mammals.png"
    et.render(str(out_path), tree_style=ts, w=width, dpi=120)
    return Path(out_path)


def draw_tree_ete(tree, title, show_bootstrap=False, with_silhouettes=True, width=1000):
    """ETE3 で系統樹を描画し、ノートブックにインライン表示する。"""
    png = render_ete_to_png(tree, title, show_bootstrap=show_bootstrap,
                            with_silhouettes=with_silhouettes, width=width)
    display(IPyImage(filename=str(png)))


# === セクション 9:まずはシルエットなしのシンプルな樹 ===
draw_tree_ete(nj_tree,
              "哺乳類 cytb の NJ 系統樹(外群=ヒトで Rooting)",
              with_silhouettes=False)

### 5.6 シルエット付きの系統樹

PhyloPic のシルエット画像を末端に並べ、より直感的に読める樹を作ります。
シルエットは系統色で塗り分けます。

> PhyloPic ([phylopic.org](https://www.phylopic.org/)) は生物のシルエットを Creative Commons ライセンスで配布しているプロジェクトです。
> 帰属情報は `mammals/phylopic_mammals/phylopic_attribution.tsv` を参照。

In [ ]:
# セクション 9 で定義した ETE3 ツールキットを使い、シルエット付きで描画する。
# 後のセクション(ML 系統樹)でも同じ名前で呼び出せるよう、薄いラッパーを用意しておく。
def draw_tree_with_silhouettes(tree, title, xlabel=None, show_bootstrap=False):
    """系統樹を PhyloPic シルエット付きで描画する(ETE3)。

    xlabel は後方互換のため受け取るが、ETE3 では枝長スケールバーが自動表示される。
    """
    draw_tree_ete(tree, title, show_bootstrap=show_bootstrap, with_silhouettes=True)


draw_tree_with_silhouettes(nj_tree, "哺乳類 cytb の NJ 系統樹(PhyloPic シルエット付き)")

### 5.7 別の方法（UPGMA）と比較してみる

**UPGMA** はシンプルに「距離が最も近いペアを順次まとめていく」だけの方法です。UPGMA系統樹はNJ系統樹と違って「必ず末端の位置が揃う」という特徴があります。これは、**すべての系統で進化速度が一定** という強い仮定を置いたためで、この仮定が成立する条件においては必ず正しい系統樹が得られます。ただ、現実にはこのような条件が成立することは稀です。

NJ と UPGMA を並べて、どこが違うか観察してみましょう。

In [ ]:
upgma_tree = constructor.upgma(dm)
for clade in upgma_tree.get_nonterminals():
    clade.name = None

# ETE3 でそれぞれ PNG に描き出し、matplotlib で横並びに表示する
tmp = Path(tempfile.gettempdir())
png_nj = render_ete_to_png(nj_tree, "NJ (Neighbor Joining)",
                           with_silhouettes=False, out_path=tmp / "cmp_nj_mammals.png")
png_upgma = render_ete_to_png(upgma_tree, "UPGMA",
                              with_silhouettes=False, out_path=tmp / "cmp_upgma_mammals.png")

fig, axes = plt.subplots(1, 2, figsize=(20, 16))
for ax, png in zip(axes, [png_nj, png_upgma]):
    ax.imshow(mpimg.imread(png))
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. 本格編 — MAFFT でアラインメントし、IQ-TREE で最尤系統樹を推定

ここからは、研究現場で実際に使われている **業界標準ツール** で同じデータを解析し直します。

| ツール | 役割 | 入門編との違い |
| --- | --- | --- |
| **MAFFT** | マルチプルアラインメント | 配列を端で切るだけでなく、必要に応じて **ギャップ(挿入・欠失)** を入れて精密に揃える |
| **IQ-TREE** | 最尤法による系統樹推定 | 距離法ではなく、塩基置換モデルに基づく **確率的推論**。**ブートストラップ** で各枝の信頼度も算出 |

これらは Python ライブラリではなく **独立した実行ファイル(コマンドラインツール)** です。

### 6.1 MAFFT と IQ-TREE のインストール

環境に応じて以下のいずれかを実行してください(初回のみ)。

```bash
# Google Colab・Ubuntu/Debian
!apt-get install -y mafft iqtree

# conda 環境(より新しい IQ-TREE が手に入る)
conda install -c bioconda mafft iqtree

# macOS (Homebrew)
brew install mafft iqtree
```

> Note: 環境によって IQ-TREE のコマンド名は `iqtree`, `iqtree2`, `iqtree3` のいずれかです。
> 以下のコードでは自動検出し、バージョンに応じてオプションも切り替えます。

In [ ]:
# Colab で直接インストールしたいときは下のコメントを外す
# !apt-get install -y mafft iqtree

def find_command(candidates):
    for cmd in candidates:
        path = shutil.which(cmd)
        if path:
            return cmd, path
    return None, None

mafft_cmd, mafft_path = find_command(["mafft"])
iqtree_cmd, iqtree_path = find_command(["iqtree3", "iqtree2", "iqtree"])

print(f"MAFFT  : {mafft_path or '見つかりません'}")
print(f"IQ-TREE: {iqtree_path or '見つかりません'} (コマンド名: {iqtree_cmd})")

if not (mafft_cmd and iqtree_cmd):
    print("\n⚠ どちらかが見つからない場合は、上のセルに従ってインストールしてください。")
    print("  以下の MAFFT / IQ-TREE のセクションはスキップされ、NJ 系統樹のみ得られます。")

### 6.2 MAFFT で配列をアラインメント

**MAFFT** は速くて精度の高いマルチプルアラインメントツール(京都大学・加藤らが開発)です。
`--auto` オプションで最適なアルゴリズムを自動選択します。

In [ ]:
# 整理済み ID で FASTA を再生成(MAFFT の入力に使う)
stripped_fasta = WORK_DIR / "mammals_cytb_stripped.fasta"
clean_records = list(SeqIO.parse(FASTA_PATH, "fasta"))
for rec in clean_records:
    rec.id = rec.id.split("|")[0]
    rec.description = ""
SeqIO.write(clean_records, stripped_fasta, "fasta")
print(f"ID 整理済み FASTA: {stripped_fasta}")

aligned_path = WORK_DIR / "mammals_cytb_aligned.fasta"

if mafft_cmd:
    print("\nMAFFT を実行中(数秒で終わります)...")
    with open(aligned_path, "w") as f:
        subprocess.run(
            [mafft_cmd, "--auto", "--quiet", str(stripped_fasta)],
            stdout=f, stderr=subprocess.PIPE, check=True, text=True,
        )
    print(f"完了: {aligned_path}")
else:
    print("MAFFT がインストールされていません。スキップします。")

### 6.3 アラインメント結果を確認

MAFFT は挿入・欠失(ギャップ `-`)を入れて配列をきれいに揃えます。

In [ ]:
if aligned_path.exists():
    mafft_alignment = AlignIO.read(aligned_path, "fasta")
    print(f"配列数              : {len(mafft_alignment)}")
    print(f"カラム数(MAFFT後)  : {mafft_alignment.get_alignment_length()}")
    print(f"カラム数(切り詰めのみ): {alignment.get_alignment_length()}")
    diff = mafft_alignment.get_alignment_length() - alignment.get_alignment_length()
    print(f"  → MAFFT は {diff:+d} カラム分のギャップを追加")
    
    print("\n最初の60カラム × 5配列:")
    for rec in mafft_alignment[:5]:
        print(f"  {rec.id:30s} {str(rec.seq)[:60]}")
else:
    print("MAFFT 結果がないため、このセルはスキップします。")
    mafft_alignment = None

### 6.4 保存性プロファイル（どの位置がどれだけ変わったか）

各カラムで「最も多い塩基がどれだけ多数派か」 = **保存性スコア** を計算します。

- 保存率の **高い** 領域 = タンパク質の機能的に重要な部位
- 保存率の **低い** 領域 = 中立進化に近く、種ごとに多様な部位

In [ ]:
if mafft_alignment is not None:
    aln_array = np.array([list(str(rec.seq).upper()) for rec in mafft_alignment])
    n_seqs, n_cols = aln_array.shape
    
    conservation = np.zeros(n_cols)
    for j in range(n_cols):
        col = aln_array[:, j]
        col_no_gap = col[col != "-"]
        if len(col_no_gap) > 0:
            _, counts = np.unique(col_no_gap, return_counts=True)
            conservation[j] = counts.max() / n_seqs
    
    window = 30
    smoothed = np.convolve(conservation, np.ones(window)/window, mode="valid")
    
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(range(len(smoothed)), smoothed, color="steelblue", linewidth=1.5)
    ax.axhline(0.9, color="red", linestyle="--", alpha=0.5, label="保存率 90%")
    ax.axhline(0.5, color="orange", linestyle="--", alpha=0.5, label="保存率 50%")
    ax.set_xlabel(f"アラインメント上のカラム位置({window}カラム移動平均)")
    ax.set_ylabel("保存性スコア")
    ax.set_title(f"cytb 遺伝子の保存性プロファイル({n_seqs}種、{n_cols}カラム)")
    ax.set_ylim(0, 1.05)
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    print(f"\n保存率 ≥ 90% のカラム: {(conservation >= 0.9).sum()} / {n_cols} "
          f"({100*(conservation >= 0.9).mean():.1f}%)")
    print(f"保存率 < 50% のカラム: {(conservation < 0.5).sum()} / {n_cols} "
          f"({100*(conservation < 0.5).mean():.1f}%)")
else:
    print("MAFFT 結果がないため、このセルはスキップします。")

### 6.5 最尤法と IQ-TREE

**最尤法 (Maximum Likelihood, ML)** は、データを生み出す確率モデルに基づいて、
「観察された配列を最もよく説明する系統樹」を統計的に探す方法です。

| | 距離法 (NJ) | 最尤法 (IQ-TREE) |
| --- | --- | --- |
| 入力 | 距離行列(配列の情報を要約) | アラインメント(全情報) |
| 塩基置換モデル | なし、または最初に固定 | **データから最適なモデルを自動選択** |
| 計算量 | 軽い(数秒) | 重い(38種で数十秒〜2 分) |
| 信頼度 | 通常はなし | **ブートストラップ**で各枝の支持率を算出 |

**IQ-TREE のオプション**

- `-s` : 入力アラインメント
- `-m MFP` : ModelFinder Plus(最適な塩基置換モデルを自動選択)
- `-B 1000` (v2) / `-bb 1000` (v1) : UltraFast Bootstrap を 1000 回
- `-T AUTO` (v2) / `-nt AUTO` (v1) : スレッド数の自動設定
- `-redo` : 既存の出力を上書き

**ブートストラップ** とは、アラインメントのカラムをランダムに重複抽出し直して系統樹を 1000 回作り直し、
各枝が何 % の試行で再現されたかを「**支持率**」として返す仕組みです。

In [ ]:
ml_treefile = Path(str(aligned_path) + ".treefile")

if iqtree_cmd and aligned_path.exists():
    # IQ-TREE のバージョンを検出してオプションを切り替える
    help_out = subprocess.run([iqtree_cmd, "--help"], capture_output=True, text=True)
    help_text = help_out.stdout + help_out.stderr
    is_modern = ("-B NUM" in help_text) or ("--prefix" in help_text)
    
    bootstrap_args = ["-B", "1000"] if is_modern else ["-bb", "1000"]
    threads_args = ["-T", "AUTO"] if is_modern else ["-nt", "AUTO"]
    print(f"IQ-TREE バージョン: {'v2/v3 系' if is_modern else 'v1 系'}")
    
    print(f"IQ-TREE を実行中(30秒〜2分)...")
    proc = subprocess.run(
        [iqtree_cmd, "-s", str(aligned_path), "-m", "MFP",
         *bootstrap_args, *threads_args, "-redo"],
        capture_output=True, text=True,
    )
    if proc.returncode == 0:
        print("IQ-TREE 完了")
        print("\n生成されたファイル:")
        for f in sorted(WORK_DIR.glob("mammals_cytb_aligned.fasta.*")):
            print(f"  {f.name}")
    else:
        print(f"IQ-TREE がエラーで終了しました:\n{proc.stderr[-500:]}")
else:
    print("IQ-TREE またはアラインメントが利用できないため、スキップします。")

### 6.6 採用されたモデルを確認

ModelFinder は何十種類もの塩基置換モデルから AIC/BIC で最良のものを選びます。

In [ ]:
report_file = Path(str(aligned_path) + ".iqtree")
if report_file.exists():
    text = report_file.read_text()
    for keyword in ["Best-fit model", "Model of substitution"]:
        for line in text.splitlines():
            if keyword in line:
                print(line.strip())
                break
    for keyword in ["Log-likelihood of the tree", "Unconstrained log-likelihood",
                    "Akaike information criterion (AIC) score",
                    "Bayesian information criterion (BIC) score"]:
        for line in text.splitlines():
            if keyword in line:
                print(line.strip())
                break
else:
    print("IQ-TREE レポートがないため、このセルはスキップします。")

### 6.7 ML 系統樹の読み込み

IQ-TREE の出力 `.treefile`(Newick 形式)を Biopython で読み込み、**外群(ヒト)** で根づけします。
Newick の内部ノードラベルには **ブートストラップ支持率 (0〜100)** が格納されています。

In [ ]:
if ml_treefile.exists():
    ml_tree = Phylo.read(ml_treefile, "newick")
    ml_tree.root_with_outgroup("Homo_sapiens")
    
    # Newick の内部ノードラベル(=ブートストラップ値の文字列)を confidence に移す
    for clade in ml_tree.get_nonterminals():
        if clade.name:
            try:
                clade.confidence = float(clade.name)
            except ValueError:
                pass
        clade.name = None
    
    bs_values = [c.confidence for c in ml_tree.get_nonterminals() if c.confidence is not None]
    print(f"ML 系統樹を読み込みました")
    print(f"  末端: {ml_tree.count_terminals()}")
    if bs_values:
        print(f"  ブートストラップ支持率: 平均 {np.mean(bs_values):.1f}, "
              f"範囲 [{min(bs_values):.0f}, {max(bs_values):.0f}]")
        print(f"  ≥ 95(強い支持)の枝: {sum(b >= 95 for b in bs_values)} / {len(bs_values)}")
else:
    print("ML 系統樹ファイルがないため、このセルはスキップします。")
    ml_tree = None

### 6.8 ML 系統樹をブートストラップ支持率付きで描画

各内部ノードにブートストラップ値を表示します。

- 🟢 緑 (≥ 95): 強い支持(その枝はほぼ確実)
- 🟠 橙 (70–94): 中程度の支持
- 🔴 赤 (< 70): 弱い支持(その枝は信頼できない)

In [ ]:
if ml_tree is not None:
    draw_tree_with_silhouettes(
        ml_tree,
        title="哺乳類 cytb の ML 系統樹(IQ-TREE + UltraFast Bootstrap × 1000)",
        xlabel="塩基置換数 / サイト",
        show_bootstrap=True,
    )
else:
    print("ML 系統樹がないため、このセルはスキップします。")

### 6.9 NJ と ML を並べて比較

In [ ]:
if ml_tree is not None:
    tmp = Path(tempfile.gettempdir())
    png_nj = render_ete_to_png(
        nj_tree, "NJ(切り詰めアラインメント + 同一性距離)",
        with_silhouettes=False, out_path=tmp / "cmp2_nj_mammals.png")
    png_ml = render_ete_to_png(
        ml_tree, "ML(MAFFT + IQ-TREE 最尤法 + UFBoot 1000)",
        show_bootstrap=True, with_silhouettes=False, out_path=tmp / "cmp2_ml_mammals.png")

    fig, axes = plt.subplots(1, 2, figsize=(20, 16))
    for ax, png in zip(axes, [png_nj, png_ml]):
        ax.imshow(mpimg.imread(png))
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("ML 系統樹がないため比較できません。")

### 6.10 分子時計で分岐年代を推定する（IQ-TREE の--date）

ここまでの系統樹では、枝の長さは **塩基置換数/サイト**（どれだけ変化したか）でした。
分子時計（molecular clock）の考え方を使うと、これを **時間（年代）** に変換できます。

- 「置換はおおよそ一定の速さで蓄積する」と仮定する(分子時計仮説)
- 化石などから分かっている1つの分岐年代（較正点, calibration）を手がかりに、全体の進化速度を求める
- その速度で、ほかのすべての分岐の年代を逆算する

ここでは化石記録から知られている
> **ヒゲクジラ(Mysticeti)とハクジラ(Odontoceti)の分岐 ≒ 3500 万年前(35 Ma)**

を較正点として分岐年代を推定します。すべての現生種は「現在(= 0)」に位置するとし(--date-tip 0)、ヒゲクジラとハクジラの **共通祖先(MRCA)** の年代を-35（現在から3500万年前）に固定します。較正ファイルは「2種以上の種名をカンマ区切り + 年代」で MRCA を指定します:`Balaena_mysticetus,Eubalaena_japonica,Physeter_macrocephalus,Orcinus_orca  -35`

**注目ポイント**
- クジラ類とカバの分岐はクジラの祖先が陸から水へ戻り始めたタイミングに当たります。今回の解析結果から推定した分岐時期と、初期のクジラ化石（パキケトゥスなど、約5000万年前=始新世）は一致しているでしょうか。
- クジラ類や鰭脚類、海牛類が、海に進出したタイミングは年代軸上で一致しているでしょうか。それとも独立しているでしょうか？

In [ ]:
import platform
import urllib.request
import tarfile

def download_iqtree_lsd2(dest_dir, ver="2.4.0"):
    """LSD2 入りの公式 IQ-TREE 静的バイナリを取得し、実行ファイルのパスを返す。"""
    sysname, mach = platform.system(), platform.machine().lower()
    if sysname == "Linux":
        asset = f"iqtree-{ver}-Linux-{'arm' if ('arm' in mach or 'aarch' in mach) else 'intel'}.tar.gz"
    elif sysname == "Darwin":
        asset = f"iqtree-{ver}-macOS.tar.gz"
    else:
        print("  自動ダウンロードは Linux / macOS のみ対応です。手動で IQ-TREE>=2.2 を入れてください。")
        return None
    dest = Path(dest_dir); dest.mkdir(exist_ok=True)
    tgz = dest / asset
    if not tgz.exists():
        print(f"  LSD2 入り IQ-TREE をダウンロード中: {asset}")
        urllib.request.urlretrieve(
            f"https://github.com/iqtree/iqtree2/releases/download/v{ver}/{asset}", tgz)
    with tarfile.open(tgz) as t:
        t.extractall(dest)
    for name in ("iqtree2", "iqtree3"):
        for exe in dest.rglob(name):
            exe.chmod(0o755)
            return str(exe)
    return None

# 較正ファイル(ヒゲクジラ + ハクジラの MRCA = 35 Ma 前)を書き出す
calib_file = WORK_DIR / "calibration_whale.txt"
calib_file.write_text(
    "Balaena_mysticetus,Eubalaena_japonica,Physeter_macrocephalus,Orcinus_orca -35\n")
print(f"較正ファイル: {calib_file}")
print(f"  内容: {calib_file.read_text().strip()}")

def _best_model():
    rep = Path(str(aligned_path) + ".iqtree")
    if rep.exists():
        for line in rep.read_text().splitlines():
            if "Best-fit model according to" in line:
                return line.split(":")[-1].strip()
    return "GTR+F+G4"

def run_dating(cmd, prefix):
    """固定した ML 系統樹の上で年代推定だけを行う(-te で topology 固定、-o でヒト外群により rooting)。"""
    return subprocess.run(
        [cmd, "-s", str(aligned_path), "-te", str(ml_treefile),
         "-m", _best_model(), "--date", str(calib_file), "--date-tip", "0",
         "-o", "Homo_sapiens", "--prefix", str(prefix), "-T", "AUTO", "--redo"],
        capture_output=True, text=True)

dating_prefix = None
if iqtree_cmd and aligned_path.exists() and ml_treefile.exists():
    dating_prefix = WORK_DIR / "mammals_dating"
    print(f"\n採用モデル: {_best_model()}")
    print("分子時計分析(IQ-TREE --date / LSD2)を実行中…(数分かかります)")
    proc = run_dating(iqtree_cmd, dating_prefix)
    if "not compiled with LSD2" in (proc.stdout + proc.stderr):
        print("  既存の IQ-TREE は LSD2 非対応 → 公式バイナリ(LSD2 入り)を取得して再実行します")
        dating_cmd = download_iqtree_lsd2(WORK_DIR / "iqtree_lsd2")
        if dating_cmd:
            proc = run_dating(dating_cmd, dating_prefix)
    if proc.returncode == 0:
        lsd_log = Path(str(dating_prefix) + ".timetree.lsd")
        if lsd_log.exists():
            for line in lsd_log.read_text().splitlines():
                if "rate" in line and "tMRCA" in line:
                    print("  LSD2 推定:", line.strip())
        print("分子時計分析 完了 →", Path(str(dating_prefix) + ".timetree.nex").name)
    else:
        print("分子時計分析が失敗しました:\n", proc.stderr[-600:])
        dating_prefix = None
else:
    print("アラインメント / ML 系統樹がないため、分子時計分析はスキップします。")

In [ ]:
import re
import io

dating_nex = Path(str(dating_prefix) + ".timetree.nex") if dating_prefix else None
if dating_nex and dating_nex.exists():
    # 1) 時間樹(枝の長さ = 百万年)を読み込む(NEXUS の [&date=...] 注釈を除去)
    nex_text = dating_nex.read_text()
    tree_line = [l for l in nex_text.splitlines() if l.strip().lower().startswith("tree ")][0]
    newick = re.sub(r'\[&date="[^"]*"\]', "", tree_line.split("=", 1)[1].strip())
    time_tree = Phylo.read(io.StringIO(newick), "newick")

    # 2) 主要な分岐年代(節の年代 = その節から葉までの時間)を表示
    def node_age(clade):
        d, c = 0.0, clade
        while c.clades:
            c = c.clades[0]
            d += c.branch_length or 0.0
        return d
    def mrca_age(taxa):
        return node_age(time_tree.common_ancestor(taxa))

    events = [
        ("ヒゲクジラ × ハクジラ(較正点)", ["Balaena_mysticetus", "Orcinus_orca"]),
        ("クジラ類 × カバ(陸→水)",       ["Balaena_mysticetus", "Hippopotamus_amphibius"]),
        ("ヒゲクジラ類の多様化",           ["Balaena_mysticetus", "Balaenoptera_musculus"]),
        ("鰭脚類(アザラシ・アシカ)",      ["Phoca_vitulina", "Odobenus_rosmarus"]),
        ("海牛類(マナティー・ジュゴン)",  ["Trichechus_manatus", "Dugong_dugon"]),
        ("全体の根(ヒトとの分岐)",        [l.name for l in time_tree.get_terminals()]),
    ]
    print("推定された主な分岐年代(百万年前, Ma):")
    for label, taxa in events:
        print(f"  {label:26s} : {mrca_age(taxa):5.1f} Ma")

    # 3) 時間樹(chronogram)を ETE3 で描画(枝長 = 時間、赤数字 = 分岐年代 Ma)
    def draw_chronogram(bio_tree, title):
        et = bio_to_ete(bio_tree)
        def age(n):
            d, c = 0.0, n
            while c.children:
                c = c.children[0]
                d += c.dist
            return d
        def layout(node):
            if node.is_leaf():
                color = id_to_color.get(node.name, "#000000")
                ns = NodeStyle(); ns["hz_line_color"] = color
                ns["hz_line_width"] = 2; ns["size"] = 0
                node.set_style(ns)
                sil = colored_silhouette(node.name, color)
                if sil:
                    node.add_face(ImgFace(sil[0], width=sil[1], height=36),
                                  column=0, position="aligned")
                node.add_face(TextFace("  " + ja_map.get(node.name, node.name),
                                       fsize=10, ftype=JP_FONT, fgcolor=color),
                              column=1, position="aligned")
            else:
                ns = NodeStyle(); ns["hz_line_color"] = "#888888"
                ns["vt_line_color"] = "#888888"; ns["size"] = 0
                node.set_style(ns)
                if node.support and node.support >= 80:
                    a = age(node)
                    if a >= 3:
                        node.add_face(TextFace(f"{a:.0f} ", fsize=7, fgcolor="#b22222"),
                                      column=0, position="branch-top")
        ts = TreeStyle(); ts.show_leaf_name = False; ts.layout_fn = layout
        ts.show_scale = True
        ts.title.add_face(TextFace(title, fsize=12, ftype=JP_FONT), column=0)
        for cl, c in clade_to_color.items():
            ts.legend.add_face(RectFace(14, 14, c, c), column=0)
            ts.legend.add_face(TextFace("  " + cl + "  ", fsize=9), column=1)
        ts.legend_position = 4
        out = Path(tempfile.gettempdir()) / "chronogram_mammals.png"
        et.render(str(out), tree_style=ts, w=1000, dpi=120)
        display(IPyImage(filename=str(out)))

    draw_chronogram(
        time_tree,
        "哺乳類 cytb の分子時計(較正: ヒゲ/ハクジラ分岐 = 35 Ma)  枝の長さ = 時間(百万年)・赤数字 = 分岐年代 Ma")
else:
    print("時間樹がないため、このセルはスキップします。")

### 6.11 形質マッピング — 系統樹に「生態」を重ねる

系統樹はそれ自体が「進化の歴史」ですが、そこに **形質(生態的な特徴)** を重ねると、
「ある特徴が進化の中で何回・どこで現れたか」が見えてきます。

ここでは各種に2つの形質を割り当て、系統樹の右側に色のマス目で並べます(**形質マッピング**)。

- **生息環境**：海生 / 半水生 / 陸生
- **食性**：肉食 / 草食 / 雑食

注目してほしいのは、**同じ「海生」が系統樹のあちこちに散らばっている** ことです。
クジラ類・海牛類・鰭脚類・ラッコは、別々の陸生祖先から **独立に海へ進出**（収斂進化）しました。
また、同じ海生でも **クジラは肉食・海牛は草食** と食性が分かれている点にも注目しましょう。

In [ ]:
# === 各種の形質データ(生息環境・食性)===
_ids = set(metadata["id"])
_SEMI = {"Phoca_vitulina", "Mirounga_leonina", "Halichoerus_grypus", "Eumetopias_jubatus",
         "Zalophus_californianus", "Odobenus_rosmarus",        # 鰭脚類
         "Lutra_lutra", "Ursus_maritimus", "Hippopotamus_amphibius"}  # カワウソ・ホッキョクグマ・カバ
_TERR = {"Ursus_arctos", "Loxodonta_africana", "Procavia_capensis", "Homo_sapiens"}
_HERB = {"Trichechus_manatus", "Dugong_dugon", "Hippopotamus_amphibius",
         "Loxodonta_africana", "Procavia_capensis"}
_OMNI = {"Ursus_arctos", "Homo_sapiens"}

habitat = {i: ("陸生" if i in _TERR else "半水生" if i in _SEMI else "海生") for i in _ids}
diet    = {i: ("草食" if i in _HERB else "雑食" if i in _OMNI else "肉食") for i in _ids}

# 形質ごとの色(直感的に:水色系=水辺、赤=肉食、緑=草食)
HABITAT_COLOR = {"海生": "#08519c", "半水生": "#6baed6", "陸生": "#d9a066"}
DIET_COLOR    = {"肉食": "#c0392b", "草食": "#27ae60", "雑食": "#f39c12"}

def draw_tree_with_traits(tree, title):
    """系統樹の右側に、各種の生息環境・食性を色のマス目でマッピングして描画する。"""
    et = bio_to_ete(tree)

    def layout(node):
        if node.is_leaf():
            color = id_to_color.get(node.name, "#000000")
            ns = NodeStyle(); ns["hz_line_color"] = color
            ns["hz_line_width"] = 2; ns["size"] = 0
            node.set_style(ns)
            sil = colored_silhouette(node.name, color)
            if sil:
                node.add_face(ImgFace(sil[0], width=sil[1], height=36),
                              column=0, position="aligned")
            node.add_face(TextFace("  " + ja_map.get(node.name, node.name) + "  ",
                                   fsize=10, ftype=JP_FONT, fgcolor=color),
                          column=1, position="aligned")
            # 形質のマス目:column 2 = 生息環境、column 3 = 食性
            hf = RectFace(20, 20, "#333333", HABITAT_COLOR.get(habitat.get(node.name), "#ffffff"))
            hf.margin_left = 4; hf.margin_right = 4; hf.margin_top = 1; hf.margin_bottom = 1
            node.add_face(hf, column=2, position="aligned")
            df = RectFace(20, 20, "#333333", DIET_COLOR.get(diet.get(node.name), "#ffffff"))
            df.margin_right = 4; df.margin_top = 1; df.margin_bottom = 1
            node.add_face(df, column=3, position="aligned")
        else:
            ns = NodeStyle(); ns["hz_line_color"] = "#888888"
            ns["vt_line_color"] = "#888888"; ns["size"] = 0
            node.set_style(ns)

    ts = TreeStyle(); ts.show_leaf_name = False; ts.layout_fn = layout
    ts.show_scale = False
    ts.title.add_face(TextFace(title, fsize=13, ftype=JP_FONT), column=0)
    # 凡例(系統 + 生息環境 + 食性)
    ts.legend.add_face(TextFace(" 生息環境 ", fsize=10, ftype=JP_FONT), column=0)
    ts.legend.add_face(TextFace("", fsize=10), column=1)
    for k, c in HABITAT_COLOR.items():
        ts.legend.add_face(RectFace(16, 16, "#333333", c), column=0)
        ts.legend.add_face(TextFace(" " + k + "  ", fsize=10, ftype=JP_FONT), column=1)
    ts.legend.add_face(TextFace(" 食性 ", fsize=10, ftype=JP_FONT), column=0)
    ts.legend.add_face(TextFace("", fsize=10), column=1)
    for k, c in DIET_COLOR.items():
        ts.legend.add_face(RectFace(16, 16, "#333333", c), column=0)
        ts.legend.add_face(TextFace(" " + k + "  ", fsize=10, ftype=JP_FONT), column=1)
    ts.legend_position = 4

    out = Path(tempfile.gettempdir()) / "trait_map_mammals.png"
    et.render(str(out), tree_style=ts, w=1100, dpi=120)
    display(IPyImage(filename=str(out)))

# ML 系統樹があればそれを、なければ NJ 系統樹を使う
_trait_tree = ml_tree if ("ml_tree" in dir() and ml_tree is not None) else nj_tree
draw_tree_with_traits(
    _trait_tree,
    "哺乳類の系統樹に形質をマッピング（左マス = 生息環境, 右マス = 食性）")

**読み取りのヒント**

- **「海生」が複数の系統に分散** … 水生への適応は1回きりではなく、**クジラ類・海牛類・鰭脚類・ラッコ** で独立に起きました(**収斂進化**)。
  枝をたどると、それぞれ別の陸生祖先から海へ入ったことが分かります。
- **海の草食動物** … 海生のほとんどは肉食ですが、**海牛類(マナティー・ジュゴン)だけは草食**。海草を主食とする数少ない海生哺乳類です。
- **カバ** … クジラの最も近い親戚(姉妹群)ですが、**半水生・草食**。完全水生・肉食のクジラとは生態が大きく異なります。
- **発展(祖先形質復元)** … 内部の枝に祖先の状態を推定して塗ると、「進化のどの時点で海に入ったか」をより明確に図示できます。
  最節約法(Fitch法)や最尤法で推定でき、`ete3` や専用ツール(`phytools` など)で実装できます。

### 6.12 祖先形質復元 — 「いつ海に入ったか」を推定する

形質マッピング(12.11)では現生種の形質を樹の **先端** に並べました。次は一歩進んで、
**内部の枝(= 過去の祖先)がどんな状態だったか** を推定します。これを **祖先形質復元
(ancestral state reconstruction)** と呼びます。

ここでは最も基本的な **最節約法(Fitch のアルゴリズム)** を使います。

- 各種を「**水生**(海生・半水生)/ **陸生**」の2状態に単純化する
- 「状態の変化(進化)の回数が最小になる」ように、各内部ノードの状態を割り当てる
- 枝を推定状態で塗り分け、**陸生 → 水生 への移行(★)** がどこで起きたかを見る

In [ ]:
def reconstruct_fitch(ete_tree, state_of):
    """最節約法(Fitch)で各ノードの祖先状態を推定し、{node: state} を返す。"""
    # down-pass(葉 → 根):各ノードで「ありうる状態の集合」を求める
    down = {}
    for node in ete_tree.traverse("postorder"):
        if node.is_leaf():
            down[node] = {state_of[node.name]}
        else:
            sets = [down[c] for c in node.children]
            inter = set.intersection(*sets)
            down[node] = inter if inter else set.union(*sets)   # 共通があれば交差、なければ和集合
    # up-pass(根 → 葉):親と整合するように1つの状態に確定する
    final = {}
    for node in ete_tree.traverse("preorder"):
        if node.is_root():
            final[node] = "陸生" if "陸生" in down[node] else sorted(down[node])[0]
        else:
            parent = final[node.up]
            final[node] = parent if parent in down[node] else sorted(down[node])[0]
    return final

def draw_ancestral_states(tree, title):
    """枝を推定された生息環境(水生/陸生)で塗り分けて描画する。"""
    et = bio_to_ete(tree)
    # 生息環境を2状態に単純化(海生・半水生 → 水生)
    binary = {i: ("水生" if habitat.get(i) in ("海生", "半水生") else "陸生") for i in habitat}
    final = reconstruct_fitch(et, binary)
    STATE_COLOR = {"水生": "#08519c", "陸生": "#d9a066"}

    # 変化(進化)が起きた枝と、陸生→水生 の移行を数える
    changes = [n for n in et.traverse() if not n.is_root() and final[n] != final[n.up]]
    aquatic_origins = [n for n in changes if final[n] == "水生" and final[n.up] == "陸生"]

    def layout(node):
        col = STATE_COLOR[final[node]]
        ns = NodeStyle(); ns["hz_line_color"] = col; ns["hz_line_width"] = 3
        ns["vt_line_color"] = "#bbbbbb"; ns["size"] = 0
        node.set_style(ns)
        if node in aquatic_origins:                       # 陸生→水生 の移行に印
            node.add_face(TextFace("★", fsize=12, fgcolor="#08519c"),
                          column=0, position="branch-top")
        if node.is_leaf():
            sil = colored_silhouette(node.name, col)
            if sil:
                node.add_face(ImgFace(sil[0], width=sil[1], height=34),
                              column=0, position="aligned")
            node.add_face(TextFace("  " + ja_map.get(node.name, node.name),
                                   fsize=10, ftype=JP_FONT, fgcolor=col),
                          column=1, position="aligned")

    ts = TreeStyle(); ts.show_leaf_name = False; ts.layout_fn = layout; ts.show_scale = False
    ts.title.add_face(TextFace(title, fsize=13, ftype=JP_FONT), column=0)
    for k, c in STATE_COLOR.items():
        ts.legend.add_face(RectFace(16, 16, c, c), column=0)
        ts.legend.add_face(TextFace(" " + k + "の枝  ", fsize=10, ftype=JP_FONT), column=1)
    ts.legend.add_face(TextFace(" ★", fsize=12, fgcolor="#08519c"), column=0)
    ts.legend.add_face(TextFace(" 陸生→水生 の移行 ", fsize=10, ftype=JP_FONT), column=1)
    ts.legend_position = 4

    out = Path(tempfile.gettempdir()) / "asr_mammals.png"
    et.render(str(out), tree_style=ts, w=1000, dpi=360)
    display(IPyImage(filename=str(out)))
    print(f"推定された根(共通祖先)の生息環境 : {final[et]}")
    print(f"最節約法による変化の総数        : {len(changes)} 回")
    print(f"うち 陸生 → 水生 への移行       : {len(aquatic_origins)} 回")

_asr_tree = ml_tree if ("ml_tree" in dir() and ml_tree is not None) else nj_tree
draw_ancestral_states(_asr_tree, "祖先形質復元(最節約法):枝の色 = 推定された生息環境")

**議論のポイント**

- **根(共通祖先)は「陸生」と推定** … 外群のヒトや、樹の根もとの陸生種(ゾウ・ハイラックスなど)に引っ張られ、深い祖先は陸生と復元されます。「哺乳類はもともと陸の動物だった」という事実と整合します。
- **★(陸生→水生)の位置** … 水生への移行が起きたと推定される枝。そこから先がまとめて青(水生)に塗られます。
- このデータは海生哺乳類に偏って選んでいます(38種中34種が水生)。最節約法は「変化の回数を最小化」するため、**水生種が多いと「祖先も水生で、変化はわずか」と推定しがち** です。実際ここでは陸生→水生の移行がごく少数に復元されますが、本来はクジラ類・鰭脚類・海牛類・カワウソ/ラッコがそれぞれ独立に水生（もっと多くの移行）したと考えられています。

## 7. 考察 — 哺乳類はいかにして海に戻ったか

ML系統樹を眺めながら、哺乳類の進化について考えてみましょう。

### Q1. 海への進出は何回起きた？

「水中生活への適応(流線型の体・ヒレ・水中呼吸)」は、系統樹を見ると **少なくとも3回独立に進化** したことが分かるはずです。

| 海生グループ | 進出時期 | 陸上の最近縁種 |
| --- | --- | --- |
| **Cetacea**(クジラ目) | 約 5000 万年前 | **カバ**(Hippopotamus) |
| **Sirenia**(海牛目) | 約 5000 万年前 | **ゾウ・ハイラックス**(アフリカ獣類) |
| **Pinnipedia**(鰭脚類) | 約 3000 万年前 | **クマ・カワウソ**などの食肉目 |

これらは系統樹上で **「別々の枝」** として現れ、それぞれの近縁な陸生種と一緒のグループに入っていますね。これがまさに **収斂進化 (convergent evolution)** です。

### Q2. クジラは誰の仲間？

形態だけで分類していた時代、クジラは「特殊な独立した目」と考えられていました。
しかし 1990年代以降の分子系統解析で、**クジラの最も近い親戚はカバ** であることが明らかになりました。

→ 今回の系統樹で、**カバ** はどのグループの近くに来ていますか?
→ この枝のブートストラップ支持率はどうですか?

現在では、クジラと偶蹄目を合わせて **Cetartiodactyla(鯨偶蹄目)** と呼びます。

### Q3. ヒゲクジラとハクジラはどこで分かれた?

クジラ目は約 3500 万年前に **Mysticeti（ヒゲクジラ:濾過摂食）** と **Odontoceti（ハクジラ:エコーロケーション）** に分かれました。

→ 系統樹の中で2系統は明確に分離していますか?
→ Odontocetiの中に **カワイルカ**（Platanista, Inia, Lipotes）はどう入っていますか？実は最近の研究から、カワイルカは **海から川に何度も独立に進出** したと考えられており、単系統群ではないとされています。

### Q4. アシカ・アザラシ・セイウチの関係

鰭脚類(Pinnipedia)は3つの科に分かれます。

- **Otariidae**（アシカ科）: カリフォルニアアシカ、トド
- **Phocidae**（アザラシ科）: ゴマフアザラシ、ハイイロアザラシ、ミナミゾウアザラシ
- **Odobenidae**（セイウチ科）: セイウチ

→ 系統樹の中で3グループに分かれていますか？セイウチはアシカとアザラシのどちらに近いですか?

### Q5. クマの遥かなる旅路

**ホッキョクグマ(Ursus maritimus)** と **ヒグマ(Ursus arctos)** は同じ属で、ごく最近（約15万年前)分かれた近縁種であることが分かっています（なんと、この２種は交雑可能です）。

→ 樹の中で2種は隣合っていますか？枝の長さはとても短いはずです。

### Q6. アフリカ獣類（Paenungulata）の絆

**マナティー・ジュゴン**（Sirenia）、**ゾウ**、**ハイラックス**——
見た目はまったく違うこの3グループは、すべて **アフリカで進化した古い系統**（Paenungulata）で互いに最も近縁です。

→ 樹の中で5種が1つのまとまりを作っていますか?この発見もまた、分子系統学が形態学を覆した例です。

### Q7. 最新の研究を眺めてみよう
[TimeTree](http://www.timetree.org/) や [OneZoom](http://www.onezoom.org/) で本物の哺乳類系統樹を見てみよう。

→**アフリカ獣類**（Afrotheria）の関係を見てみましょう。近年の系統ゲノミクスで特に進化の解明が進んだグループの一つです。